## UKB and CHARLS data are not public open-access

## We randomly sample metabolite/blood biomarker concentrations to establish toy data for demonstrating how to use MetFoundation

## ⚠️ IMPORTANT DISCLAIMER ⚠️

### The generated data below are randomly sampled FAKE DATA
- **These data are completely unrelated to the original real data**
- Only maintains consistency in data format, feature names, and metadata structure
- Numeric values are randomly generated based on the ranges of original data
- **Cannot be used for any scientific research or analysis**
- Only for code demonstration and tool usage examples

## Import Required Libraries

In [1]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Scanpy version: {sc.__version__}")

Libraries imported successfully!
NumPy version: 2.0.2
Pandas version: 2.3.3
Scanpy version: 1.11.5


## Define Functions for Generating Fake Data

In [2]:
def generate_fake_data_from_h5ad(original_file_path, n_samples=1000, random_state=42):
    """
    Read data structure from original .h5ad file and generate randomly sampled fake data
    
    Parameters:
    -----------
    original_file_path : str
        Path to the original .h5ad file
    n_samples : int
        Number of fake data samples to generate, default 1000
    random_state : int
        Random seed for reproducibility
        
    Returns:
    --------
    fake_adata : AnnData
        AnnData object containing fake data
    """
    np.random.seed(random_state)
    
    # Read original data to get structure information
    print(f"\n{'='*60}")
    print(f"Processing: {original_file_path}")
    print(f"{'='*60}")
    
    try:
        original_adata = sc.read_h5ad(original_file_path)
        print(f"✓ Original data loaded successfully")
        print(f"  - Shape: {original_adata.shape}")
        print(f"  - Features: {original_adata.var.shape[0]}")
        print(f"  - Observations metadata: {original_adata.obs.shape[1]} columns")
    except Exception as e:
        print(f"✗ Error loading file: {e}")
        return None
    
    # Get feature names
    var_names = original_adata.var_names.tolist()
    n_features = len(var_names)
    
    # Generate fake expression matrix (based on value ranges from original data)
    print(f"\n→ Generating fake expression matrix...")
    fake_X = np.zeros((n_samples, n_features))
    
    for i, feature in enumerate(var_names):
        feature_values = original_adata.X[:, i]
        if hasattr(feature_values, 'toarray'):
            feature_values = feature_values.toarray().flatten()
        else:
            feature_values = np.asarray(feature_values).flatten()
        
        # Remove NaN and Inf values
        valid_values = feature_values[np.isfinite(feature_values)]
        
        if len(valid_values) > 0:
            min_val = np.min(valid_values)
            max_val = np.max(valid_values)
            mean_val = np.mean(valid_values)
            std_val = np.std(valid_values)
            
            # Generate using normal distribution, keep within reasonable range
            fake_values = np.random.normal(mean_val, std_val, n_samples)
            fake_values = np.clip(fake_values, min_val, max_val)
        else:
            # If no valid values, generate zeros
            fake_values = np.zeros(n_samples)
        
        fake_X[:, i] = fake_values
    
    print(f"  ✓ Expression matrix generated: {fake_X.shape}")
    
    # Generate fake obs metadata
    print(f"\n→ Generating fake observation metadata...")
    fake_obs = pd.DataFrame(index=[f"sample_{i}" for i in range(n_samples)])
    
    for col in original_adata.obs.columns:
        col_data = original_adata.obs[col]
        
        if pd.api.types.is_numeric_dtype(col_data):
            # Numeric: random sampling within original range
            valid_values = col_data[pd.notna(col_data)]
            if len(valid_values) > 0:
                if pd.api.types.is_integer_dtype(col_data):
                    min_val = int(valid_values.min())
                    max_val = int(valid_values.max())
                    fake_obs[col] = np.random.randint(min_val, max_val + 1, n_samples)
                else:
                    min_val = valid_values.min()
                    max_val = valid_values.max()
                    mean_val = valid_values.mean()
                    std_val = valid_values.std()
                    fake_obs[col] = np.random.normal(mean_val, std_val, n_samples)
                    fake_obs[col] = np.clip(fake_obs[col], min_val, max_val)
            else:
                fake_obs[col] = np.nan
        elif pd.api.types.is_categorical_dtype(col_data) or pd.api.types.is_object_dtype(col_data):
            # Categorical or string: randomly choose from original categories
            unique_values = col_data.dropna().unique()
            if len(unique_values) > 0:
                fake_obs[col] = np.random.choice(unique_values, n_samples)
            else:
                fake_obs[col] = None
        elif pd.api.types.is_bool_dtype(col_data):
            # Boolean: random True/False
            fake_obs[col] = np.random.choice([True, False], n_samples)
        else:
            # Other types: set to None
            fake_obs[col] = None
    
    print(f"  ✓ Observation metadata generated: {fake_obs.shape}")
    
    # Copy var metadata (feature information remains unchanged)
    fake_var = original_adata.var.copy()
    print(f"  ✓ Variable metadata copied: {fake_var.shape}")
    
    # Create new AnnData object
    fake_adata = ad.AnnData(X=fake_X, obs=fake_obs, var=fake_var)
    
    # If original data has uns attributes, copy them as well
    if len(original_adata.uns) > 0:
        fake_adata.uns = original_adata.uns.copy()
        print(f"  ✓ Unstructured metadata copied")
    
    print(f"\n✓ Fake data generation completed!")
    print(f"  - Final shape: {fake_adata.shape}")
    
    return fake_adata

## Scan and List All .h5ad Files

In [3]:
# Get current notebook directory
current_dir = Path.cwd()
data_dir = current_dir  # Currently in Data directory

print(f"Current directory: {current_dir}")
print(f"Data directory: {data_dir}")

# Find all .h5ad files
h5ad_files = list(data_dir.glob("**/*.h5ad"))

print(f"\n{'='*60}")
print(f"Found {len(h5ad_files)} .h5ad files:")
print(f"{'='*60}")
for i, file_path in enumerate(h5ad_files, 1):
    rel_path = file_path.relative_to(data_dir)
    print(f"{i}. {rel_path}")
    print(f"   Full path: {file_path}")

Current directory: /datahome/comp/ericteam/csyuxu/MetFoundation/Data
Data directory: /datahome/comp/ericteam/csyuxu/MetFoundation/Data

Found 4 .h5ad files:
1. CHARLS/charls_2015_adata.h5ad
   Full path: /datahome/comp/ericteam/csyuxu/MetFoundation/Data/CHARLS/charls_2015_adata.h5ad
2. CHARLS/charls_2011_adata.h5ad
   Full path: /datahome/comp/ericteam/csyuxu/MetFoundation/Data/CHARLS/charls_2011_adata.h5ad
3. UKB_NMR/val.h5ad
   Full path: /datahome/comp/ericteam/csyuxu/MetFoundation/Data/UKB_NMR/val.h5ad
4. UKB_Blood/val.h5ad
   Full path: /datahome/comp/ericteam/csyuxu/MetFoundation/Data/UKB_Blood/val.h5ad


## Generate Fake Data for All Files

In [4]:
# Generate fake data for each .h5ad file
n_fake_samples = 1000  # Generate 1000 fake samples
fake_data_results = {}

print(f"\n{'#'*60}")
print(f"# Starting Fake Data Generation")
print(f"# Number of samples per file: {n_fake_samples}")
print(f"{'#'*60}\n")

for file_path in h5ad_files:
    try:
        # Generate fake data
        fake_adata = generate_fake_data_from_h5ad(
            str(file_path), 
            n_samples=n_fake_samples, 
            random_state=42
        )
        
        if fake_adata is not None:
            # Save fake data (named as fake_xxx.h5ad)
            rel_path = file_path.relative_to(data_dir)
            output_filename = f"fake_{file_path.name}"
            output_path = file_path.parent / output_filename
            
            fake_adata.write_h5ad(output_path)
            print(f"\n✓ Fake data saved to: {output_path.relative_to(data_dir)}")
            
            fake_data_results[str(rel_path)] = {
                'status': 'success',
                'output_file': str(output_path.relative_to(data_dir)),
                'shape': fake_adata.shape
            }
        else:
            fake_data_results[str(rel_path)] = {
                'status': 'failed',
                'error': 'Failed to generate fake data'
            }
    except Exception as e:
        print(f"\n✗ Error processing {file_path.name}: {e}")
        fake_data_results[str(file_path.relative_to(data_dir))] = {
            'status': 'error',
            'error': str(e)
        }

print(f"\n{'#'*60}")
print(f"# Fake Data Generation Summary")
print(f"{'#'*60}")


############################################################
# Starting Fake Data Generation
# Number of samples per file: 1000
############################################################


Processing: /datahome/comp/ericteam/csyuxu/MetFoundation/Data/CHARLS/charls_2015_adata.h5ad
✓ Original data loaded successfully
  - Shape: (13257, 14)
  - Features: 14
  - Observations metadata: 3 columns

→ Generating fake expression matrix...
  ✓ Expression matrix generated: (1000, 14)

→ Generating fake observation metadata...
  ✓ Observation metadata generated: (1000, 3)
  ✓ Variable metadata copied: (14, 0)

✓ Fake data generation completed!
  - Final shape: (1000, 14)

✓ Fake data saved to: CHARLS/fake_charls_2015_adata.h5ad

Processing: /datahome/comp/ericteam/csyuxu/MetFoundation/Data/CHARLS/charls_2011_adata.h5ad
✓ Original data loaded successfully
  - Shape: (11823, 14)
  - Features: 14
  - Observations metadata: 3 columns

→ Generating fake expression matrix...
  ✓ Expression matrix gen

## Summary Report

In [10]:
# Print processing results summary
print("\nProcessing Results:")
print("="*80)

success_count = 0
failed_count = 0

for original_file, result in fake_data_results.items():
    print(f"\n📁 Original: {original_file}")
    
    if result['status'] == 'success':
        print(f"   ✓ Status: SUCCESS")
        print(f"   → Fake data: {result['output_file']}")
        print(f"   → Shape: {result['shape'][0]} samples × {result['shape'][1]} features")
        success_count += 1
    else:
        print(f"   ✗ Status: FAILED")
        print(f"   → Error: {result.get('error', 'Unknown error')}")
        failed_count += 1

print("\n" + "="*80)
print(f"Total: {len(fake_data_results)} files")
print(f"  ✓ Success: {success_count}")
print(f"  ✗ Failed: {failed_count}")
print("="*80)

print("\n⚠️  REMINDER: These are FAKE DATA randomly generated for demonstration only!")
print("    Do NOT use for any real analysis or research!")


Processing Results:

📁 Original: CHARLS/charls_2015_adata.h5ad
   ✓ Status: SUCCESS
   → Fake data: CHARLS/fake_charls_2015_adata.h5ad
   → Shape: 1000 samples × 14 features

📁 Original: CHARLS/charls_2011_adata.h5ad
   ✓ Status: SUCCESS
   → Fake data: CHARLS/fake_charls_2011_adata.h5ad
   → Shape: 1000 samples × 14 features

📁 Original: UKB_NMR/val.h5ad
   ✓ Status: SUCCESS
   → Fake data: UKB_NMR/fake_val.h5ad
   → Shape: 1000 samples × 107 features

📁 Original: UKB_Blood/val.h5ad
   ✓ Status: SUCCESS
   → Fake data: UKB_Blood/fake_val.h5ad
   → Shape: 1000 samples × 14 features

Total: 4 files
  ✓ Success: 4
  ✗ Failed: 0

⚠️  REMINDER: These are FAKE DATA randomly generated for demonstration only!
    Do NOT use for any real analysis or research!


## Verification: Load and Check One Fake Data File

In [11]:
# Verification: load and check one fake data file
if len(h5ad_files) > 0:
    # Select the first generated fake data file
    first_file = h5ad_files[0]
    fake_file_path = first_file.parent / f"fake_{first_file.name}"
    
    if fake_file_path.exists():
        print(f"Verifying fake data file: {fake_file_path.relative_to(data_dir)}")
        print("="*60)
        
        test_adata = sc.read_h5ad(fake_file_path)
        
        print(f"\n📊 Data Overview:")
        print(f"  - Shape: {test_adata.shape}")
        print(f"  - Number of observations: {test_adata.n_obs}")
        print(f"  - Number of variables: {test_adata.n_vars}")
        
        print(f"\n📋 Observation metadata (obs):")
        print(f"  - Columns: {list(test_adata.obs.columns)}")
        print(f"  - First 3 samples:")
        print(test_adata.obs.head(3))
        
        print(f"\n🧬 Variable metadata (var):")
        print(f"  - Columns: {list(test_adata.var.columns)}")
        print(f"  - First 5 features:")
        print(test_adata.var.head(5))
        
        print(f"\n📈 Expression matrix statistics:")
        print(f"  - Min value: {test_adata.X.min():.4f}")
        print(f"  - Max value: {test_adata.X.max():.4f}")
        print(f"  - Mean value: {test_adata.X.mean():.4f}")
        print(f"  - Median value: {np.median(test_adata.X):.4f}")
        
        print(f"\n✓ Verification completed successfully!")
    else:
        print(f"Fake data file not found: {fake_file_path}")
else:
    print("No .h5ad files found for verification.")

Verifying fake data file: CHARLS/fake_charls_2015_adata.h5ad

📊 Data Overview:
  - Shape: (1000, 14)
  - Number of observations: 1000
  - Number of variables: 14

📋 Observation metadata (obs):
  - Columns: ['Sex', 'Chronological Age', 'BMI']
  - First 3 samples:
          Sex  Chronological Age        BMI
sample_0    1          57.721148  23.417447
sample_1    1          57.859316  23.851134
sample_2    1          83.071240  24.230724

🧬 Variable metadata (var):
  - Columns: []
  - First 5 features:
Empty DataFrame
Columns: []
Index: [White blood cell (leukocyte) count, Haemoglobin concentration, Haematocrit percentage, Mean corpuscular volume, Platelet count]

📈 Expression matrix statistics:
  - Min value: 0.1000
  - Max value: 569.9523
  - Mean value: 55.6295
  - Median value: 11.0280

✓ Verification completed successfully!
